In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/CW_Folder_UG/Code')

from utils import *

In [ ]:
ROOT        = '/content/drive/MyDrive/CW_Folder_UG/'
TRAIN_IMGS  = f'{ROOT}/CW_Dataset/train'
TRAIN_LBL   = f'{ROOT}/CW_Dataset/train/train_labels.txt'
TEST_IMGS   = f'{ROOT}/CW_Dataset/test'
TEST_LBL    = f'{ROOT}/CW_Dataset/test/test_labels.txt'
MODELS_DIR  = f'{ROOT}/Models'
os.makedirs(MODELS_DIR, exist_ok=True)

In [ ]:
train_paths, train_labels   = load_dataset(TRAIN_IMGS, TRAIN_LBL)
test_paths, test_labels     = load_dataset(TEST_IMGS,  TEST_LBL)

print(f'Train samples : {len(train_paths)}')
print(f'Test  samples : {len(test_paths)}')

print('Train class distribution:', Counter(train_labels))
print('Test  class distribution:', Counter(test_labels))

In [ ]:
counts = Counter(train_labels)
fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']
bars = ax.bar([AGE_LABELS[k] for k in range(4)],
              [counts[k] for k in range(4)], color=colors)
ax.set_title('Training Set Class Distribution')
ax.set_ylabel('Count')
for bar, k in zip(bars, range(4)):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 30, str(counts[k]),
            ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

class_weights = get_class_weights(train_labels)
print('Class weights:', class_weights)

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(15, 12))
for row, label in enumerate(range(4)):
    idxs = random.sample([i for i, l in enumerate(train_labels) if l == label], 5)
    for col, idx in enumerate(idxs):
        ax = axes[row][col]
        ax.imshow(load_image(train_paths[idx]))
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(AGE_LABELS[label], fontsize=12)
            
plt.suptitle('5 samples per age group (training set)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
heights, widths = [], []
for p in random.sample(train_paths, min(500, len(train_paths))):
    img = cv2.imread(p)
    if img is not None:
        h, w = img.shape[:2]
        heights.append(h); widths.append(w)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))

a1.hist(widths,  bins=30, color='steelblue')
a1.set_title('Image Widths')

a2.hist(heights, bins=30, color='coral')
a2.set_title('Image Heights')

plt.suptitle('Resolution Distribution (500-sample subset)')
plt.tight_layout()
plt.show()

print(f"Width: min:{min(widths)}  max:{max(widths)}  median:{int(np.median(widths))}")
print(f"Height: min:{min(heights)}  max:{max(heights)}  median:{int(np.median(heights))}")

In [ ]:
# Save so other notebooks can load without re-parsing
import pickle
with open(f'{ROOT}/Code/data_splits.pkl', 'wb') as f:
    pickle.dump({
        'train_paths':  train_paths,
        'train_labels': train_labels,
        'test_paths':   test_paths,
        'test_labels':  test_labels
    }, f)
print("Saved data_splits.pkl")